In [1]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000) ## 800GB?

from calitp.tables import tbl
# from calitp import query_sql
import calitp.magics

import shared_utils
# import utils

from siuba import *
import pandas as pd
import geopandas as gpd
# import shapely

import datetime as dt
import time
# from zoneinfo import ZoneInfo

# import gcsfs
# fs = gcsfs.GCSFileSystem()

from tqdm import tqdm_notebook
from tqdm.notebook import trange, tqdm

/opt/conda/lib/python3.10/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.10.2-CAPI-1.16.0) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


# Prioritizing Notices -- which should we provide guidance for first?

In [2]:
tbl.views.validation_fact_daily_feed_notices() >> head(3)

,feed_key,date,validation_created_at,validation_deleted_at,calitp_itp_id,calitp_url_number,code,severity,csvRowNumber,oldCsvRowNumber,...,routeLongName,routeDesc,stopId,stopName,serviceIdA,serviceIdB,departureTime,arrivalTime,parentStation,parentStopName
0,-3540410672069212686,2022-04-28,2022-04-28,2022-04-29,105,0,same_name_and_description_for_route,ERROR,11,None,...,None,Route 12,None,None,None,None,None,None,None,None
1,1273942275370395959,2022-06-02,2022-06-02,2022-06-03,231,0,same_name_and_description_for_route,ERROR,32,None,...,None,60-Hwy126,None,None,None,None,None,None,None,None
2,4493221564916387693,2022-05-03,2022-05-03,2022-05-04,361,0,same_name_and_description_for_route,ERROR,4,None,...,None,Route 2,None,None,None,None,None,None,None,None


In [3]:
tbl.views.validation_fact_daily_feed_codes() >> head(3)

,feed_key,code,date,n_notices,diff_n_notices,is_error_resolved,is_error_introduced
0,2590856768872525679,empty_file,2021-12-19,3,NaN,False,True
1,-5246656847780582136,invalid_url,2022-02-04,3,0.0,False,False
2,-5246656847780582136,invalid_url,2022-01-21,3,0.0,False,False


In [6]:
code_rankings = (tbl.views.validation_fact_daily_feed_codes()
     >> filter(_.date == '2022-06-01')
     >> filter(_.n_notices > 0)
     >> count(_.code)
    ) >> collect()

In [7]:
code_rankings

,code,n
0,unknown_file,181
1,unknown_column,134
2,duplicate_route_name,48
3,decreasing_or_equal_shape_distance,47
4,feed_expiration_date,46
5,too_fast_travel,34
6,stop_too_far_from_trip_shape,23
7,leading_or_trailing_whitespaces,22
8,unused_shape,21
9,route_short_name_too_long,20


## duplicate_route_name

In [8]:
dup_routes = (tbl.views.validation_fact_daily_feed_notices()
     >> filter(_.date == '2022-06-01')
     >> filter(_.code == 'duplicate_route_name')
    ) >> collect()

In [10]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(dup_routes >> head(10)) 

,feed_key,date,validation_created_at,validation_deleted_at,calitp_itp_id,calitp_url_number,code,severity,csvRowNumber,oldCsvRowNumber,newCsvRowNumber,csvRowNumberA,csvRowNumberB,rowLength,headerCount,previousCsvRowNumber,filename,fieldName,fieldName1,fieldName2,fieldValue,fieldValue1,fieldValue2,index,shapeDistTraveled,shapePtSequence,prevCsvRowNumber,prevShapeDistTraveled,prevShapePtSequence,duplicatedField,suggestedExpirationDate,suggestedExpirationDate.localDate,suggestedExpirationDate.localDate.year,suggestedExpirationDate.localDate.month,suggestedExpirationDate.localDate.day,stopSequence,specifiedField,prevStopTimeDistTraveled,prevStopSequence,speedkmh,firstStopSequence,lastStopSequence,stopShapeThresholdMeters,blockId,intersection,expectedLocationType,locationType,parentCsvRowNumber,parentLocationType,entityCount,fieldType,childFieldName,parentFieldName,childFilename,parentFilename,fareId,previousFareId,shapeId,routeId,currentDate,feedEndDate,routeColor,routeTextColor,tripId,tripIdA,tripIdB,routeShortName,routeLongName,routeDesc,stopId,stopName,serviceIdA,serviceIdB,departureTime,arrivalTime,parentStation,parentStopName
0,-3696369716674726447,2022-06-01,2022-06-01,2022-06-02,232,0,duplicate_route_name,WARNING,45,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_long_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,10422,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,4164903538710312962,2022-06-01,2022-06-01,2022-06-02,182,0,duplicate_route_name,WARNING,76,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_long_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,237-13159,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,4164903538710312962,2022-06-01,2022-06-01,2022-06-02,182,0,duplicate_route_name,WARNING,38,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_long_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,125-13159,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,4164903538710312962,2022-06-01,2022-06-01,2022-06-02,182,0,duplicate_route_name,WARNING,5,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_long_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,14-13159,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,4164903538710312962,2022-06-01,2022-06-01,2022-06-02,182,0,duplicate_route_name,WARNING,41,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_long_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,130-13159,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
5,-5693386325383800078,2022-06-01,2022-06-01,2022-06-02,239,0,duplicate_route_name,WARNING,8,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,route_short_name,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,746,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
6,4164903538710312962,2022-06-01,2022-06-01,2022-06-02,182,0,duplicate_route_name,WARNING,78,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None

In [28]:
dup_routes >> filter(_.calitp_itp_id != 182) >> count(_.calitp_itp_id, _.calitp_url_number) >> arrange(-_.n)

,calitp_itp_id,calitp_url_number,n
26,232,0,53
7,61,0,43
23,200,0,18
46,482,0,17
33,293,0,14
35,295,0,10
43,360,1,10
15,142,0,9
27,235,0,9
18,167,1,5


In [13]:
routes_on_date = tbl.views.gtfs_schedule_fact_daily_feed_routes() >> filter(_.date == '2022-06-01')

In [16]:
tbl_routes = tbl.views.gtfs_schedule_dim_routes() >> inner_join(_, routes_on_date, on = 'route_key')

In [29]:
def show_rt_counts(itp_id, url_number):
    tbl_routes = tbl.views.gtfs_schedule_dim_routes() >> inner_join(_, routes_on_date, on = 'route_key')
    this_feed = tbl_routes >> filter(_.calitp_itp_id == itp_id, _.calitp_url_number == url_number)
    print('count of short names:')
    display(this_feed >> count(_.route_short_name) >> arrange(-_.n))
    print('count of long names:')
    display(this_feed >> count(_.route_long_name) >> arrange(-_.n))
    return

In [30]:
show_rt_counts(61, 0)

count of short names:


,route_short_name,n
0,16,2
1,21,2
2,9,2
3,99X,2
4,11,2


count of long names:


,route_long_name,n
0,Concord BART/Walnut Creek BART,2
1,Concord BART/Pleasant Hill BART,2
2,Martinez Amtrak/Concord BART,2
3,DVC/Concord BART,2
4,Walnut Creek BART/San Ramon,2


In [31]:
show_rt_counts(482, 0)

count of short names:


,route_short_name,n
0,Rte 2,4
1,Rte 3,3
2,ADV,3
3,PINE,3
4,Rte 5,2


count of long names:


,route_long_name,n
0,Adventure Trolley,3
1,Pinecrest,3
2,Route 2 - To Sierra Village,2
3,Route 3 - To Sonora-Jamestown,1
4,Route 2 - To Sonora,1


In [32]:
show_rt_counts(182, 0)

count of short names:


,route_short_name,n
0,None,3
1,246,1
2,South Bay Dodger Stadium Express,1
3,33,1
4,154,1


count of long names:


,route_long_name,n
0,Metro Local Line,100
1,Metro Express Line,6
2,Metro Rapid Line,3
3,None,2
4,Metro J Line (Silver) 910/950,1


In [33]:
show_rt_counts(293, 0)

count of short names:


,route_short_name,n
0,24X,1
1,3,1
2,2530,1
3,2410,1
4,2430,1


count of long names:


,route_long_name,n
0,Goleta Valley Jr. High,5
1,Dos Pueblos High School,5
2,San Marcos High School,4
3,La Colina Jr. High,4
4,Goleta Express,1


In [34]:
show_rt_counts(295, 0)

count of short names:


,route_short_name,n
0,12,1
1,101,1
2,102,1
3,622,1
4,200,1


count of long names:


,route_long_name,n
0,La Mesa Jr High,4
1,Saugus High,3
2,Golden Valley High,3
3,Century City,2
4,Warner Center,2
